# 01 -- Subprocess et shell

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- executer des commandes systeme avec `subprocess.run` ;
- capturer stdout, stderr et le code de retour ;
- passer des arguments de facon securisee (sans injection shell) ;
- utiliser `Popen` pour le controle avance (pipes, streaming) ;
- gerer les timeouts et les erreurs ;
- chainer des commandes (equivalent du pipe shell) ;
- remplacer les anciens `os.system` et `os.popen`.

## Prerequis -- ce que vous devez maitriser

- les fonctions Python et la gestion d'exceptions ;
- les bases du terminal Linux/macOS (ls, cat, grep, echo) ;

Ce que ce notebook **ne couvrira pas** :

- les sockets reseau (voir `02_Sockets_tcp_udp.ipynb`) ;
- HTTP (voir `03_HTTP_avec_httpx.ipynb`).

## Plan

1. Pourquoi subprocess
2. `subprocess.run` -- la base
3. Capturer la sortie
4. Gerer les erreurs
5. Arguments et securite
6. Timeout
7. Entree stdin
8. `Popen` -- controle avance
9. Chainer des commandes (pipes)
10. Variables d'environnement
11. Patterns pratiques
12. Synthese
13. Exercices

---

## 1. Pourquoi subprocess

Le module `subprocess` est la facon moderne et securisee d'executer des commandes systeme depuis Python. Il remplace `os.system`, `os.popen`, et les fonctions du module `commands` (supprime en Python 3).

In [ ]:
import subprocess

---

## 2. `subprocess.run` -- la base

`run` execute une commande et attend qu'elle se termine.

In [ ]:
result = subprocess.run(['echo', 'Hello from Python'])
result

In [ ]:
result.returncode  # 0 = succes

**Toujours passer les arguments en liste**, pas en chaine. C'est plus sur et plus portable.

---

## 3. Capturer la sortie

In [ ]:
result = subprocess.run(['uname', '-a'], capture_output=True, text=True)
result.stdout

In [ ]:
result.stderr

`capture_output=True` est equivalent a `stdout=subprocess.PIPE, stderr=subprocess.PIPE`. `text=True` decode la sortie en `str` (au lieu de `bytes`).

In [ ]:
# Alternative : check_output
output = subprocess.check_output(['hostname'], text=True)
output.strip()

---

## 4. Gerer les erreurs

In [ ]:
result = subprocess.run(['ls', '/dossier_inexistant'], capture_output=True, text=True)
result.returncode  # non zero = erreur

In [ ]:
result.stderr

### `check=True` pour lever une exception

In [ ]:
try:
    subprocess.run(['ls', '/dossier_inexistant'], check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as e:
    print(f'Commande echouee (code {e.returncode})')
    print(f'stderr : {e.stderr}')

---

## 5. Arguments et securite

### Jamais `shell=True` avec des donnees utilisateur

In [ ]:
# DANGEREUX : injection shell possible
# subprocess.run(f'ls {user_input}', shell=True)  # NE PAS FAIRE

# SECURISE : arguments en liste
user_input = '/tmp'
subprocess.run(['ls', user_input], capture_output=True, text=True).stdout[:200]

Avec `shell=True`, `subprocess` passe la commande a `/bin/sh`. Un utilisateur malveillant peut injecter `; rm -rf /`. Avec une liste, chaque element est un argument distinct, pas d'interpretation shell.

### shlex.split pour parser une chaine

In [ ]:
import shlex
shlex.split('grep -r "hello world" /tmp')

---

## 6. Timeout

In [ ]:
try:
    subprocess.run(['sleep', '10'], timeout=2)
except subprocess.TimeoutExpired:
    print('Commande interrompue apres 2 secondes')

---

## 7. Entree stdin

In [ ]:
result = subprocess.run(
    ['wc', '-w'],
    input='un deux trois quatre cinq',
    capture_output=True, text=True,
)
result.stdout.strip()  # 5 mots

---

## 8. `Popen` -- controle avance

`Popen` lance un processus sans attendre sa fin. Utile pour le streaming ou les pipes.

In [ ]:
proc = subprocess.Popen(
    ['ls', '-la', '/tmp'],
    stdout=subprocess.PIPE, text=True,
)
for line in proc.stdout:
    if 'python' in line.lower():
        print(line.strip())
proc.wait()

### communicate pour envoyer et recevoir

In [ ]:
proc = subprocess.Popen(
    ['sort'],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True,
)
stdout, stderr = proc.communicate(input='banane\npomme\ncerise\n')
print(stdout)

---

## 9. Chainer des commandes (pipes)

Equivalent de `ls -la | grep .py | wc -l` :

In [ ]:
p1 = subprocess.Popen(['ls', '-la', '/tmp'], stdout=subprocess.PIPE)
p2 = subprocess.Popen(['grep', 'python'], stdin=p1.stdout, stdout=subprocess.PIPE, text=True)
p1.stdout.close()  # permet a p1 de recevoir SIGPIPE
output = p2.communicate()[0]
print(output)

---

## 10. Variables d'environnement

In [ ]:
import os

env = os.environ.copy()
env['MY_VAR'] = 'hello'
result = subprocess.run(['env'], capture_output=True, text=True, env=env)
lines = [l for l in result.stdout.splitlines() if 'MY_VAR' in l]
lines

---

## 11. Patterns pratiques

### Executer un script Python enfant

In [ ]:
import sys
result = subprocess.run(
    [sys.executable, '-c', 'import sys; print(sys.version)'],
    capture_output=True, text=True,
)
result.stdout.strip()

### Verifier si une commande existe

In [ ]:
import shutil
shutil.which('git')  # retourne le chemin ou None

### Lancer une commande en arriere-plan

In [ ]:
# proc = subprocess.Popen(['long_running_command'])
# ... faire autre chose ...
# proc.wait()  # ou proc.poll() pour verifier sans bloquer

---

## Synthese

| Fonction | Usage |
|---|---|
| `subprocess.run` | Executer et attendre, usage general |
| `subprocess.check_output` | Capturer la sortie, lever si erreur |
| `subprocess.Popen` | Controle avance, pipes, streaming |
| `shlex.split` | Parser une chaine de commande |
| `shutil.which` | Verifier l'existence d'une commande |

### Regles a retenir

- **Toujours passer les arguments en liste**, pas en chaine.
- **Jamais `shell=True`** avec des donnees non fiables.
- **`check=True`** pour detecter les erreurs automatiquement.
- **`text=True`** pour travailler avec des `str` au lieu de `bytes`.
- **`Popen`** pour le streaming et les pipes.

---

## Exercices

### Exercice 1 -- Lister les processus

Ecrivez une fonction `list_python_processes()` qui utilise `ps aux` et `grep` pour trouver les processus Python en cours. Retournez une liste de lignes.

In [ ]:
import subprocess

def list_python_processes() -> list[str]:
    ...

<details>
<summary>Voir la correction</summary>

```python
import subprocess

def list_python_processes() -> list[str]:
    result = subprocess.run(
        ['ps', 'aux'], capture_output=True, text=True, check=True
    )
    return [line for line in result.stdout.splitlines() if 'python' in line.lower()]

for line in list_python_processes():
    print(line)
```

</details>

### Exercice 2 -- Ping avec timeout

Ecrivez une fonction `ping_host(host, count=3, timeout=5)` qui execute `ping -c count host` avec un timeout. Retournez True si le host repond, False sinon.

In [ ]:
def ping_host(host: str, count: int = 3, timeout: int = 5) -> bool:
    ...

<details>
<summary>Voir la correction</summary>

```python
import subprocess

def ping_host(host: str, count: int = 3, timeout: int = 5) -> bool:
    try:
        subprocess.run(
            ['ping', '-c', str(count), host],
            capture_output=True, check=True, timeout=timeout,
        )
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        return False

print(ping_host('localhost'))
```

</details>

### Exercice 3 -- Pipeline de commandes

Ecrivez une fonction `count_files(directory, extension)` qui chaine `find directory -name '*.ext'` et `wc -l` pour compter les fichiers. Utilisez `Popen` pour les pipes.

In [ ]:
def count_files(directory: str, extension: str) -> int:
    ...

<details>
<summary>Voir la correction</summary>

```python
import subprocess

def count_files(directory: str, extension: str) -> int:
    p1 = subprocess.Popen(
        ['find', directory, '-name', f'*.{extension}'],
        stdout=subprocess.PIPE,
    )
    p2 = subprocess.Popen(
        ['wc', '-l'],
        stdin=p1.stdout, stdout=subprocess.PIPE, text=True,
    )
    p1.stdout.close()
    output, _ = p2.communicate()
    return int(output.strip())

print(count_files('/tmp', 'py'))
```

</details>

### Exercice 4 -- Informations systeme

Ecrivez une fonction `system_info()` qui retourne un dictionnaire avec : hostname, OS, kernel, nb CPUs, memoire totale. Utilisez des commandes systeme.

In [ ]:
def system_info() -> dict[str, str]:
    ...

<details>
<summary>Voir la correction</summary>

```python
import subprocess

def system_info() -> dict[str, str]:
    def run(cmd):
        return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

    return {
        'hostname': run(['hostname']),
        'os': run(['uname', '-s']),
        'kernel': run(['uname', '-r']),
        'cpus': run(['nproc']),
        'arch': run(['uname', '-m']),
    }

system_info()
```

</details>

---

## Ressources externes

- [Documentation subprocess](https://docs.python.org/3/library/subprocess.html)
- [PEP 324 -- subprocess](https://peps.python.org/pep-0324/)
- [Real Python -- subprocess](https://realpython.com/python-subprocess/)